In [1]:
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
import json, random, itertools, os, gc, statistics as st
import seqeval
import re

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans
from dataclasses import dataclass, field

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def _get(d: Dict[str, Any], k: str, default=None):
    return d.get(k, default)

@dataclass
class NERMetricsCollector:
    overall_rows: List[Dict[str, Any]] = field(default_factory=list)
    label_rows:   List[Dict[str, Any]] = field(default_factory=list)
    meta:         Dict[str, Any] = field(default_factory=dict)  # ex.: nome do modelo, dataset, etc.

    def record(self, split_name: Any, metrics: Dict[str, Any], extras: Optional[Dict[str, Any]] = None):
        """
        Registra os resultados de um split.
        - split_name: pode ser string, int, tupla... será convertido para string
        - metrics: dict retornado pelo seu train/eval
        - extras: (opcional) dict com metadados (seed, versão, etc.)
        """
        split_str = str(split_name)

        # ------------------------------
        # Tabela 1: métricas gerais
        # ------------------------------
        overall = {
            "split": split_str,
            "eval_loss":                _get(metrics, "eval_loss"),
            "overall_precision":        _get(metrics, "eval_overall_precision"),
            "overall_recall":           _get(metrics, "eval_overall_recall"),
            "overall_f1":               _get(metrics, "eval_overall_f1"),
            "overall_accuracy":         _get(metrics, "eval_overall_accuracy"),
            "f1_micro":                 _get(metrics, "eval_f1_micro"),
            "f1_macro":                 _get(metrics, "eval_f1_macro"),
            "f1_weighted":              _get(metrics, "eval_f1_weighted"),
            "runtime_s":                _get(metrics, "eval_runtime"),
            "samples_per_sec":          _get(metrics, "eval_samples_per_second"),
            "steps_per_sec":            _get(metrics, "eval_steps_per_second"),
            "epoch":                    _get(metrics, "epoch"),
        }
        
        self.overall_rows.append(overall)

        # ------------------------------
        # Tabela 2: métricas por rótulo
        # ------------------------------
        # Regra: qualquer entrada do dict que seja outro dict contendo
        # precision/recall/f1/number é tratada como rótulo.
        for k, v in metrics.items():
            if isinstance(v, dict) and {"precision", "recall", "f1", "number"} <= set(v.keys()):
                self.label_rows.append({
                    "split": split_str,
                    "label": k.replace("eval_", ""),  # remove prefixo "eval_" para ficar limpo
                    "precision": v["precision"],
                    "recall":    v["recall"],
                    "f1":        v["f1"],
                    "support":   v["number"],
                })

    # Comentário: retorna DataFrames prontos para inspeção ou export
    def to_dataframes(self):
        df_overall = pd.DataFrame(self.overall_rows)
        df_labels  = pd.DataFrame(self.label_rows)
        return df_overall, df_labels

    # Comentário: exporta dois CSVs (UTF-8 com BOM para abrir liso no Excel)
    def to_csv(self, base_name: str = "ner"):
        df_overall, df_labels = self.to_dataframes()
        df_overall.to_csv(f"{base_name}_overall_metrics.csv", index=False, encoding="utf-8-sig")
        df_labels.to_csv(f"{base_name}_label_metrics.csv", index=False, encoding="utf-8-sig")
        return f"{base_name}_overall_metrics.csv", f"{base_name}_label_metrics.csv"


# --- Cria (ou reaproveita) um coletor global ---
if "ner_collector" not in globals():
    ner_collector = NERMetricsCollector()

# Configuração e Verificação Inicial

In [ ]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

CSV_PATH = Path("../data/all_ener.csv")  # ajuste se necessário
MODEL_NAME = "albert/albert-base-v2"

df = pd.read_csv(CSV_PATH)

In [4]:
df

,-DOCSTART-,O
0,NaN,O
1,Check,O
2,the,O
3,appropriate,O
4,box,O
...,...,...
415363,this,O
415364,Summary,O
415365,Prospectus,O
415366,.,O


In [5]:
NER_MARK = "<BIO-COMMA>" 

In [6]:
from datasets import Dataset, DatasetDict, Features, Sequence, Value

In [7]:
def _read_conllish_csv(df: pd.DataFrame):
    """Lê DataFrame com colunas ['-DOCSTART-', 'O'] e fecha sentenças por linha vazia ou pontuação terminal."""
    sents, tags = [], []
    cur_toks, cur_tags = [], []

    SENT_END = {".", "!", "?"}

    for _, row in df.iterrows():
        tok, lab = row["-DOCSTART-"], row["O"]

        # linha vazia / NaN => fecha sentença
        if pd.isna(tok) or str(tok).strip() == "":
            if cur_toks:
                sents.append(cur_toks); tags.append(cur_tags)
                cur_toks, cur_tags = [], []
            continue

        tok = str(tok)
        lab = str(lab)

        cur_toks.append(tok)
        cur_tags.append(lab)

        # fecha sentença se encontrar pontuação terminal
        if tok in SENT_END:
            sents.append(cur_toks); tags.append(cur_tags)
            cur_toks, cur_tags = [], []

    # flush final (última sentença sem ponto)
    if cur_toks:
        sents.append(cur_toks); tags.append(cur_tags)

    return {"tokens": sents, "ner_tags": tags}


def build_ener_dataset(
    df: pd.DataFrame,
    val_size: float = 0.15,
    test_size: float = 0.15,
    seed: int = 42,
    add_columns: bool = True,
    corpus_name: str = "ener",
):
    data = _read_conllish_csv(df)

    features = Features({
        "tokens": Sequence(Value("string")),
        "ner_tags": Sequence(Value("string")),
    })
    ds_full = Dataset.from_dict(data, features=features)

    # mapeia labels->ids
    labels = sorted({lab for seq in data["ner_tags"] for lab in seq})
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}

    return ds_full, labels, label2id, id2label

In [8]:
data = _read_conllish_csv(df)

In [9]:
len(data["tokens"])

11737

In [10]:
data

{'tokens': [['Check', 'the', 'appropriate', 'box', ':'],
  ['Payment',
   'of',
   'Filing',
   'Fee',
   '(',
   'check',
   'the',
   'appropriate',
   'box',
   ')',
   ':'],
  ['Nuveen',
   'New',
   'York',
   'Dividend',
   'Advantage',
   'Municipal',
   'Fund',
   '(',
   'NAN',
   ')'],
  ['Nuveen',
   'New',
   'York',
   'Dividend',
   'Advantage',
   'Municipal',
   'Fund',
   '2',
   '(',
   'NXK',
   ')'],
  ['Nuveen',
   'New',
   'York',
   'Investment',
   'Quality',
   'Municipal',
   'Fund',
   ',',
   'Inc.'],
  ['Nuveen', 'New', 'York', 'Municipal', 'Value', 'Fund', ',', 'Inc.'],
  ['Nuveen',
   'New',
   'York',
   'Municipal',
   'Value',
   'Fund',
   '2',
   '(',
   'NYV',
   ')'],
  ['Nuveen',
   'New',
   'York',
   'Performance',
   'Plus',
   'Municipal',
   'Fund',
   ',',
   'Inc.'],
  ['Nuveen',
   'New',
   'York',
   'Quality',
   'Income',
   'Municipal',
   'Fund',
   ',',
   'Inc.'],
  ['Nuveen',
   'New',
   'York',
   'Select',
   'Quality',
   'M

In [11]:
ener7_ds, ener7_labels, ener7_label2id, ener7_id2label = build_ener_dataset(df, corpus_name="ener7")

In [12]:
ener7_ds

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 11737
})

In [13]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in ener7_ds["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [14]:
filtered = ener7_ds.filter(
    lambda example: "I-LOC" in example["ner_tags"]
)

print(filtered)
print(filtered[0])

Filter: 100%|██████████| 11737/11737 [00:00<00:00, 16391.63 examples/s]

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 1
})
{'tokens': ['While', 'certain', 'U.S.', 'Government-sponsored', 'agencies', '(', 'such', 'as', 'the', 'Federal', 'Home', 'Loan', 'Mortgage', 'Corporation', 'and', 'the', 'Federal', 'National', 'Mortgage', 'Association', ')', 'may', 'be', 'chartered', 'or', 'sponsored', 'by', 'acts', 'of', 'Congress', ',', 'their', 'securities', 'are', 'neither', 'issued', 'nor', 'guaranteed', 'by', 'the', 'U.S.', 'Risks', 'of', 'Repurchase', 'Agreements', 'and', 'Reverse', 'Repurchase', 'Agreements', '.'], 'ner_tags': ['O', 'O', 'I-LOCATION', 'O', 'O', 'O', 'O', 'O', 'O', 'I-GOVERNMENT', 'I-GOVERNMENT', 'I-GOVERNMENT', 'I-GOVERNMENT', 'I-GOVERNMENT', 'O', 'O', 'I-GOVERNMENT', 'I-GOVERNMENT', 'I-GOVERNMENT', 'I-GOVERNMENT', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'I-GOVERNMENT', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


In [15]:
id2label

{0: 'B-BUSINESS',
 1: 'B-GOVERNMENT',
 2: 'B-LEGISLATION/ACT',
 3: 'B-LOCATION',
 4: 'B-MISCELLANEOUS',
 5: 'B-PERSON',
 6: 'I-BUSINESS',
 7: 'I-COURT',
 8: 'I-GOVERNMENT',
 9: 'I-LEGISLATION/ACT',
 10: 'I-LOC',
 11: 'I-LOCATION',
 12: 'I-MISCELLANEOUS',
 13: 'I-PERSON',
 14: 'O',
 15: 'P'}

In [16]:
NUM_LABELS

16

In [17]:
def replace_labels(example):
    new_tags = []
    for tag in example["ner_tags"]:
        if tag == "P":
            new_tags.append("O")
        elif tag == "I-LOC":
            new_tags.append("I-LOCATION")
        else:
            new_tags.append(tag)
    example["ner_tags"] = new_tags
    return example

ener7_ds = ener7_ds.map(replace_labels)


Map: 100%|██████████| 11737/11737 [00:00<00:00, 15460.74 examples/s]


In [18]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in ener7_ds["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [19]:
id2label

{0: 'B-BUSINESS',
 1: 'B-GOVERNMENT',
 2: 'B-LEGISLATION/ACT',
 3: 'B-LOCATION',
 4: 'B-MISCELLANEOUS',
 5: 'B-PERSON',
 6: 'I-BUSINESS',
 7: 'I-COURT',
 8: 'I-GOVERNMENT',
 9: 'I-LEGISLATION/ACT',
 10: 'I-LOCATION',
 11: 'I-MISCELLANEOUS',
 12: 'I-PERSON',
 13: 'O'}

# Splits

In [20]:
def split_standard(ds: Dataset) -> DatasetDict:
    """Usa coluna trainingTest do CSV (80/20 original)."""
    if "trainingTest" not in df.columns:
        raise ValueError("CSV não contém a coluna 'trainingTest'")
    train_ids = df.loc[df["trainingTest"] == "training", SENT_COL].unique()
    test_ids = df.loc[df["trainingTest"] == "test", SENT_COL].unique()
    return DatasetDict(
        train=ds.filter(lambda ex: ex["sentence_id"] in train_ids),
        dev=ds.filter(lambda ex: ex["sentence_id"] in test_ids),
    )

In [21]:
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

In [22]:
# def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
#     lengths = np.array([len(t) for t in ds["tokens"]])
#     thr = np.percentile(lengths, 100 * (1 - top_pct))
#     mask = lengths >= thr
#     return DatasetDict(train=ds.filter(~mask), dev=ds.filter(mask))


def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
    """20 % das sentenças mais longas viram conjunto de validação (dev)."""
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))  
    mask = lengths >= thr  

    dev_idx = np.where(mask)[0].tolist()  # índices → list[int]
    train_idx = np.where(~mask)[0].tolist()

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Tamanho da sentenças


# def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
#     freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
#     rare = {w for w, c in freq.items() if c <= freq_thr}

#     def has_rare(example):
#         return any(w.lower() in rare for w in example["tokens"])

#     return DatasetDict(
#         train=ds.filter(lambda ex: not has_rare(ex)), dev=ds.filter(has_rare)
#     )


def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
    freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
    rare = {w for w, c in freq.items() if c <= freq_thr}

    keep_dev = []
    for sent in ds["tokens"]:
        print(sent)
        keep_dev.append(any(w.lower() in rare for w in sent))

    dev_idx = [i for i, x in enumerate(keep_dev) if x]
    train_idx = [i for i, x in enumerate(keep_dev) if not x]

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Raridade dos tokens

In [23]:
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [24]:
# def loc_split(
#     dataset: Dataset, pct_test: float = 0.20, ngram: int = 4, seed: int = 42
# ) -> DatasetDict:
#     """
#     Split baseado em baixa sobreposição léxica (4-gram Jaccard).
#     Teste = pct_test das sentenças com menor overlap em relação ao pool.
#     """
#     # 1. Texto plano por sentença
#     docs = [" ".join(toks) for toks in dataset["tokens"]]

#     # 2. Vetorizar 4-grams (binário)
#     vect = CountVectorizer(
#         analyzer="word", ngram_range=(ngram, ngram), binary=True
#     ).fit(docs)
#     X = vect.transform(docs)

#     # 3. Similaridade Jaccard aproximada com matriz binária
#     # Jaccard(A,B) = |A∩B|/|A∪B| = 1 - |AΔB|/|A∪B|
#     # Usamos: overlap = (A·Bᵀ) / (|A|+|B|-A·Bᵀ)
#     bin_counts = X.sum(axis=1).A1

#     # Para cada doc i, escolhemos vizinho + próximo (fast):
#     from sklearn.metrics.pairwise import cosine_similarity

#     # (cosine no binário ∝ |A∩B|)
#     sim = cosine_similarity(X, dense_output=False)
#     # Soma dos top-k overlaps (k=5) como score
#     k = 5
#     topk = np.zeros(len(dataset))
#     for i in range(sim.shape[0]):
#         row = sim.getrow(i).toarray()[0]
#         idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
#         # overlap ≈ |∩|
#         inter = row[idx] * bin_counts[i]
#         uni = bin_counts[i] + bin_counts[idx] - inter
#         topk[i] = (inter / uni).mean()

#     # 4. Ordenar por overlap crescente ⇒ mais “novos” vão p/ teste
#     order = np.argsort(topk)
#     n_test = int(len(dataset) * pct_test)
#     test_idx = order[:n_test]
#     train_idx = order[n_test:]

#     return DatasetDict(
#         {"train": dataset.select(train_idx), "test": dataset.select(test_idx)}
#     )

In [25]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [26]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [27]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [28]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    while len(test_idx) < int(pct_test*len(dataset)):
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [29]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [30]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [31]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [32]:
heur_len = heur_len_split(ener7_ds)
print("heur_len")
heur_rare = heur_rare_split(ener7_ds)
print("heur_rare")
advers = adversarial_split(ener7_ds)
print("advs")
loc = loc_split(ener7_ds)
print("loc")
semantic = semantic_cluster_split(ener7_ds)
print("semantic")
reverse = reverse_curriculum_split(ener7_ds)
print("reverse")

heur_len
heur_rare
Selecionando 2347 sentenças para teste…
  0 selecionadas…
  24 selecionadas…
  48 selecionadas…
  73 selecionadas…
  96 selecionadas…
  119 selecionadas…
  138 selecionadas…
  158 selecionadas…
  177 selecionadas…
  197 selecionadas…
  215 selecionadas…
  232 selecionadas…
  247 selecionadas…
  260 selecionadas…
  285 selecionadas…
  309 selecionadas…
  331 selecionadas…
  354 selecionadas…
  374 selecionadas…
  393 selecionadas…
  413 selecionadas…
  436 selecionadas…
  453 selecionadas…
  469 selecionadas…
  486 selecionadas…
  506 selecionadas…
  524 selecionadas…
  546 selecionadas…
  561 selecionadas…
  571 selecionadas…
  586 selecionadas…
  603 selecionadas…
  613 selecionadas…
  625 selecionadas…
  643 selecionadas…
  666 selecionadas…
  690 selecionadas…
  710 selecionadas…
  733 selecionadas…
  757 selecionadas…
  781 selecionadas…
  799 selecionadas…
  818 selecionadas…
  839 selecionadas…
  856 selecionadas…
  876 selecionadas…
  898 selecionadas…
  913 s

100%|██████████| 11737/11737 [00:00<00:00, 559326.31it/s]


semantic
reverse


# Experimentos

In [33]:
from sklearn.metrics import f1_score as skl_f1

In [34]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc"      : loc_split,
            "reverse"  : reverse_curriculum_split,
            "semantic" : semantic_cluster_split,
            "heur_len" : heur_len_split,
            "heur_rare": heur_rare_split,
            "std"      : std_split,
            "advs"     : adversarial_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = sorted({l for labels in dataset["ner_tags"] for l in labels})
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        labels_batch = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)  # <- aqui sim
            label_ids = []
            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # máscara
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[word_labels[word_idx]])
                else:
                    # marca sub-tokens; mude para `label2id[...]`
                    # se quiser repetir label em todos os sub-tokens
                    label_ids.append(
                        label2id[word_labels[word_idx]] if label_all_tokens else -100
                    )
                previous_word_idx = word_idx
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []   # p/ seqeval
        flat_preds, flat_labels = [], []   # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro    = skl_f1(flat_labels, flat_preds, average="micro",    zero_division=0)
        f1_macro    = skl_f1(flat_labels, flat_preds, average="macro",    zero_division=0)
        f1_weighted = skl_f1(flat_labels, flat_preds, average="weighted", zero_division=0)

        return {
            **seqeval_metrics,            # overall_precision / recall / f1
            "f1_micro":    f1_micro,
            "f1_macro":    f1_macro,
            "f1_weighted": f1_weighted,
        }



    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="no",
        #load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none"
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [35]:
splits = ["loc", "reverse", "semantic", "heur_len", "heur_rare",  "advs"]

In [36]:
results = {}
trainer_all = {}
s = splits[0]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(ener7_ds, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: loc


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([14]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([14, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2347/2347 [00:00<00:00, 14769.98 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_10204\17258847.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Business,Court,Government,Legislation/act,Location,Miscellaneous,Person,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.072500,0.028609,"{'precision': 0.8226744186046512, 'recall': 0.8628048780487805, 'f1': 0.8422619047619048, 'number': 328}","{'precision': 0.5, 'recall': 0.3333333333333333, 'f1': 0.4, 'number': 3}","{'precision': 0.7333333333333333, 'recall': 0.6666666666666666, 'f1': 0.6984126984126984, 'number': 33}","{'precision': 0.7142857142857143, 'recall': 0.8571428571428571, 'f1': 0.7792207792207793, 'number': 35}","{'precision': 0.8969072164948454, 'recall': 0.925531914893617, 'f1': 0.9109947643979057, 'number': 94}","{'precision': 0.7346938775510204, 'recall': 0.5413533834586466, 'f1': 0.6233766233766235, 'number': 133}","{'precision': 0.8085106382978723, 'recall': 1.0, 'f1': 0.8941176470588235, 'number': 38}",0.807576,0.802711,0.805136,0.991922,0.991922,0.673267,0.991426
2,0.016800,0.028784,"{'precision': 0.8651026392961877, 'recall': 0.899390243902439, 'f1': 0.881913303437967, 'number': 328}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 3}","{'precision': 0.8666666666666667, 'recall': 0.7878787878787878, 'f1': 0.8253968253968254, 'number': 33}","{'precision': 0.7142857142857143, 'recall': 0.8571428571428571, 'f1': 0.7792207792207793, 'number': 35}","{'precision': 0.9456521739130435, 'recall': 0.925531914893617, 'f1': 0.935483870967742, 'number': 94}","{'precision': 0.6153846153846154, 'recall': 0.7218045112781954, 'f1': 0.6643598615916955, 'number': 133}","{'precision': 0.9024390243902439, 'recall': 0.9736842105263158, 'f1': 0.9367088607594938, 'number': 38}",0.814184,0.864458,0.838568,0.992429,0.992429,0.714991,0.992542
3,0.009700,0.025755,"{'precision': 0.8592814371257484, 'recall': 0.875, 'f1': 0.8670694864048338, 'number': 328}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 3}","{'precision': 0.8125, 'recall': 0.7878787878787878, 'f1': 0.8, 'number': 33}","{'precision': 0.8333333333333334, 'recall': 0.8571428571428571, 'f1': 0.8450704225352113, 'number': 35}","{'precision': 0.8979591836734694, 'recall': 0.9361702127659575, 'f1': 0.9166666666666666, 'number': 94}","{'precision': 0.7737226277372263, 'recall': 0.7969924812030075, 'f1': 0.7851851851851852, 'number': 133}","{'precision': 0.9024390243902439, 'recall': 0.9736842105263158, 'f1': 0.9367088607594938, 'number': 38}",0.847283,0.868976,0.857993,0.994230,0.994230,0.734911,0.994176
4,0.005800,0.026737,"{'precision': 0.8666666666666667, 'recall': 0.8719512195121951, 'f1': 0.8693009118541033, 'number': 328}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 3}","{'precision': 0.8666666666666667, 'recall': 0.7878787878787878, 'f1': 0.8253968253968254, 'number': 33}","{'precision': 0.75, 'recall': 0.8571428571428571, 'f1': 0.7999999999999999, 'number': 35}","{'precision': 0.9354838709677419, 'recall': 0.925531914893617, 'f1': 0.9304812834224598, 'number': 94}","{'precision': 0.74, 'recall': 0.8345864661654135, 'f1': 0.784452296819788, 'number': 133}","{'precision': 0.925, 'recall': 0.9736842105263158, 'f1': 0.9487179487179489, 'number': 38}",0.845481,0.873494,0.859259,0.994005,0.994005,0.831309,0.994062
5,0.004100,0.027736,"{'precision': 0.8765060240963856, 'recall': 0.8871951219512195, 'f1': 0.8818181818181818, 'number': 328}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 3}","{'precision': 0.8709677419354839, 'recall': 0.8181818181818182, 'f1': 0.84375, 'number': 33}","{'precision': 0.7631578947368421, 'recall': 0.8285714285714286, 'f1': 0.7945205479452055, 'number': 35}","{'precision': 0.946236559139785, 'recall': 0.9361702127659575, 'f1': 0.9411764705882354, 'number': 94}","{'precision': 0.7737226277372263, 'recall': 0.7969924812030075, 'f1': 0.7851851851851852, 'number': 133}","{'precision': 0.9024390243902439, 'recall': 0.9736842105263158, 'f1': 0.9367088607594938, 'number': 38}",0.860741,0.875000,0.867812,0.994005,0.994005,0.830841,0.993

F1 Macro: 0.5516181902604662
F1 Micro: 0.9849938928633746
F1 Weighted: 0.9843055874075716
{'eval_loss': 0.07563649863004684, 'eval_BUSINESS': {'precision': 0.8432835820895522, 'recall': 0.8625954198473282, 'f1': 0.8528301886792453, 'number': 655}, 'eval_COURT': {'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'f1': 0.6666666666666666, 'number': 6}, 'eval_GOVERNMENT': {'precision': 0.7142857142857143, 'recall': 0.813953488372093, 'f1': 0.7608695652173914, 'number': 43}, 'eval_LEGISLATION/ACT': {'precision': 0.7916666666666666, 'recall': 0.6551724137931034, 'f1': 0.7169811320754716, 'number': 29}, 'eval_LOCATION': {'precision': 0.738255033557047, 'recall': 0.9166666666666666, 'f1': 0.8178438661710038, 'number': 120}, 'eval_MISCELLANEOUS': {'precision': 0.681592039800995, 'recall': 0.6171171171171171, 'f1': 0.6477541371158393, 'number': 222}, 'eval_PERSON': {'precision': 0.8389261744966443, 'recall': 0.8223684210526315, 'f1': 0.8305647840531561, 'number': 152}, 'eval_overal

20

In [37]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu":"0"})

In [38]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [39]:
results = {}
trainer_all = {}
s = splits[1]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(ener7_ds, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: reverse


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([14]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([14, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2347/2347 [00:00<00:00, 5050.47 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_10204\17258847.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Business,Court,Government,Legislation/act,Location,Miscellaneous,Person,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.034900,0.141538,"{'precision': 0.615727002967359, 'recall': 0.8366935483870968, 'f1': 0.7094017094017093, 'number': 496}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}","{'precision': 1.0, 'recall': 0.5294117647058824, 'f1': 0.6923076923076924, 'number': 119}","{'precision': 0.5087719298245614, 'recall': 0.5311355311355311, 'f1': 0.5197132616487455, 'number': 273}","{'precision': 0.8012820512820513, 'recall': 0.7911392405063291, 'f1': 0.7961783439490445, 'number': 158}","{'precision': 0.7954545454545454, 'recall': 0.43316831683168316, 'f1': 0.5608974358974359, 'number': 404}","{'precision': 0.8571428571428571, 'recall': 0.48484848484848486, 'f1': 0.6193548387096773, 'number': 99}",0.667813,0.626452,0.646471,0.971751,0.971751,0.466582,0.967308
2,0.005000,0.180798,"{'precision': 0.6666666666666666, 'recall': 0.8024193548387096, 'f1': 0.7282708142726441, 'number': 496}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}","{'precision': 1.0, 'recall': 0.5546218487394958, 'f1': 0.7135135135135136, 'number': 119}","{'precision': 0.5714285714285714, 'recall': 0.5714285714285714, 'f1': 0.5714285714285714, 'number': 273}","{'precision': 0.8466666666666667, 'recall': 0.8037974683544303, 'f1': 0.8246753246753248, 'number': 158}","{'precision': 0.7773279352226721, 'recall': 0.4752475247524752, 'f1': 0.5898617511520737, 'number': 404}","{'precision': 0.948051948051948, 'recall': 0.7373737373737373, 'f1': 0.8295454545454545, 'number': 99}",0.717730,0.652903,0.683784,0.972169,0.972169,0.489811,0.967948
3,0.003000,0.176510,"{'precision': 0.7038917089678511, 'recall': 0.8387096774193549, 'f1': 0.765409383624655, 'number': 496}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}","{'precision': 1.0, 'recall': 0.5630252100840336, 'f1': 0.7204301075268817, 'number': 119}","{'precision': 0.5735849056603773, 'recall': 0.5567765567765568, 'f1': 0.5650557620817843, 'number': 273}","{'precision': 0.8333333333333334, 'recall': 0.8227848101265823, 'f1': 0.8280254777070064, 'number': 158}","{'precision': 0.7198581560283688, 'recall': 0.5024752475247525, 'f1': 0.5918367346938775, 'number': 404}","{'precision': 0.9583333333333334, 'recall': 0.696969696969697, 'f1': 0.8070175438596493, 'number': 99}",0.723657,0.669032,0.695273,0.973904,0.973904,0.493237,0.970290
4,0.001600,0.175523,"{'precision': 0.7705544933078394, 'recall': 0.8125, 'f1': 0.7909715407262021, 'number': 496}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}","{'precision': 1.0, 'recall': 0.5798319327731093, 'f1': 0.7340425531914894, 'number': 119}","{'precision': 0.5547445255474452, 'recall': 0.5567765567765568, 'f1': 0.5557586837294333, 'number': 273}","{'precision': 0.8466666666666667, 'recall': 0.8037974683544303, 'f1': 0.8246753246753248, 'number': 158}","{'precision': 0.6513157894736842, 'recall': 0.4900990099009901, 'f1': 0.5593220338983051, 'number': 404}","{'precision': 0.9886363636363636, 'recall': 0.8787878787878788, 'f1': 0.9304812834224598, 'number': 99}",0.735795,0.668387,0.700473,0.974030,0.974030,0.505254,0.970542
5,0.001100,0.189810,"{'precision': 0.7462406015037594, 'recall': 0.8004032258064516, 'f1': 0.7723735408560312, 'number': 496}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1}","{'precision': 1.0, 'recall': 0.5882352941176471, 'f1': 0.7407407407407407, 'number': 119}","{'precision': 0.5539568345323741, 'recall': 0.5641025641025641, 'f1': 0.558983666061706, 'number': 273}","{'precision': 0.8424657534246576, 'recall': 0.7784810126582279, 'f1': 0.8092105263157894, 'number': 158}","{'precision': 0.6936619718309859, 'recall': 0.4876237623762376, 'f1': 0.5726744186046512, 'number': 404}","{'precision': 0.975609756097561, 'recall': 0.8080808080808081, 'f1': 0.8839779005524863, 'number': 99}",0.733477,0.658710,0.694086,0.973800

c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\s

F1 Macro: 0.32995920117345445
F1 Micro: 0.9372475320242801
F1 Weighted: 0.9251809212317753
{'eval_loss': 0.49873021245002747, 'eval_BUSINESS': {'precision': 0.538021978021978, 'recall': 0.5373134328358209, 'f1': 0.5376674719964859, 'number': 2278}, 'eval_COURT': {'precision': 0.3333333333333333, 'recall': 0.10714285714285714, 'f1': 0.16216216216216214, 'number': 28}, 'eval_GOVERNMENT': {'precision': 0.9436619718309859, 'recall': 0.31678486997635935, 'f1': 0.4743362831858407, 'number': 423}, 'eval_LEGISLATION/ACT': {'precision': 0.382051282051282, 'recall': 0.40877914951989025, 'f1': 0.3949635520212061, 'number': 729}, 'eval_LOCATION': {'precision': 0.7048951048951049, 'recall': 0.628428927680798, 'f1': 0.6644693473961766, 'number': 802}, 'eval_MISCELLANEOUS': {'precision': 0.596078431372549, 'recall': 0.37163814180929094, 'f1': 0.4578313253012048, 'number': 818}, 'eval_PERSON': {'precision': 0.7931818181818182, 'recall': 0.6763565891472868, 'f1': 0.7301255230125523, 'number': 516}, 'ev

20

In [40]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu":"0"})

In [41]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [42]:
results = {}
trainer_all = {}
s = splits[2]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(ener7_ds, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: semantic


100%|██████████| 11737/11737 [00:00<00:00, 1445857.20it/s]
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([14]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([14, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2349/2349 [00:00<00:00, 11661.84 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_10204\17258847.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Business,Court,Government,Legislation/act,Location,Miscellaneous,Person,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.069900,0.028676,"{'precision': 0.8471337579617835, 'recall': 0.8807947019867549, 'f1': 0.8636363636363636, 'number': 151}","{'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'f1': 0.6666666666666666, 'number': 3}","{'precision': 0.4583333333333333, 'recall': 0.8461538461538461, 'f1': 0.5945945945945945, 'number': 13}","{'precision': 0.7636363636363637, 'recall': 0.8571428571428571, 'f1': 0.8076923076923076, 'number': 49}","{'precision': 0.8888888888888888, 'recall': 0.9523809523809523, 'f1': 0.9195402298850575, 'number': 42}","{'precision': 0.7567567567567568, 'recall': 0.5185185185185185, 'f1': 0.6153846153846154, 'number': 54}","{'precision': 0.9021739130434783, 'recall': 0.9764705882352941, 'f1': 0.9378531073446328, 'number': 85}",0.820823,0.853904,0.837037,0.994579,0.994579,0.708629,0.994628
2,0.017400,0.024443,"{'precision': 0.8580645161290322, 'recall': 0.8807947019867549, 'f1': 0.869281045751634, 'number': 151}","{'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'f1': 0.6666666666666666, 'number': 3}","{'precision': 0.35294117647058826, 'recall': 0.9230769230769231, 'f1': 0.5106382978723405, 'number': 13}","{'precision': 0.7884615384615384, 'recall': 0.8367346938775511, 'f1': 0.8118811881188118, 'number': 49}","{'precision': 0.6964285714285714, 'recall': 0.9285714285714286, 'f1': 0.7959183673469388, 'number': 42}","{'precision': 0.7142857142857143, 'recall': 0.6481481481481481, 'f1': 0.6796116504854369, 'number': 54}","{'precision': 0.9625, 'recall': 0.9058823529411765, 'f1': 0.9333333333333333, 'number': 85}",0.790210,0.853904,0.820823,0.994052,0.994052,0.695259,0.994517
3,0.008900,0.024323,"{'precision': 0.8711656441717791, 'recall': 0.9403973509933775, 'f1': 0.9044585987261147, 'number': 151}","{'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'f1': 0.6666666666666666, 'number': 3}","{'precision': 1.0, 'recall': 0.9230769230769231, 'f1': 0.9600000000000001, 'number': 13}","{'precision': 0.8571428571428571, 'recall': 0.8571428571428571, 'f1': 0.8571428571428571, 'number': 49}","{'precision': 0.9069767441860465, 'recall': 0.9285714285714286, 'f1': 0.9176470588235294, 'number': 42}","{'precision': 0.7222222222222222, 'recall': 0.7222222222222222, 'f1': 0.7222222222222222, 'number': 54}","{'precision': 0.9222222222222223, 'recall': 0.9764705882352941, 'f1': 0.9485714285714287, 'number': 85}",0.867150,0.904282,0.885327,0.995809,0.995809,0.791158,0.995731
4,0.005300,0.031823,"{'precision': 0.86875, 'recall': 0.9205298013245033, 'f1': 0.8938906752411576, 'number': 151}","{'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'f1': 0.6666666666666666, 'number': 3}","{'precision': 0.36666666666666664, 'recall': 0.8461538461538461, 'f1': 0.5116279069767441, 'number': 13}","{'precision': 0.875, 'recall': 0.8571428571428571, 'f1': 0.8659793814432989, 'number': 49}","{'precision': 0.9318181818181818, 'recall': 0.9761904761904762, 'f1': 0.9534883720930233, 'number': 42}","{'precision': 0.5714285714285714, 'recall': 0.7407407407407407, 'f1': 0.6451612903225806, 'number': 54}","{'precision': 0.9222222222222223, 'recall': 0.9764705882352941, 'f1': 0.9485714285714287, 'number': 85}",0.804494,0.901763,0.850356,0.994052,0.994052,0.726594,0.994686
5,0.003400,0.024882,"{'precision': 0.9056603773584906, 'recall': 0.9536423841059603, 'f1': 0.9290322580645162, 'number': 151}","{'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'f1': 0.6666666666666666, 'number': 3}","{'precision': 0.6666666666666666, 'recall': 0.9230769230769231, 'f1': 0.7741935483870968, 'number': 13}","{'precision': 0.9148936170212766, 'recall': 0.8775510204081632, 'f1': 0.8958333333333333, 'number': 49}","{'precision': 0.82, 'recall': 0.9761904761904762, 'f1': 0.8913043478260869, 'number': 42}","{'precision': 

F1 Macro: 0.7334672943155827
F1 Micro: 0.9865994774169466
F1 Weighted: 0.9862825518658914
{'eval_loss': 0.08066357672214508, 'eval_BUSINESS': {'precision': 0.8543388429752066, 'recall': 0.8882921589688507, 'f1': 0.8709847288046341, 'number': 931}, 'eval_COURT': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 6}, 'eval_GOVERNMENT': {'precision': 0.6274509803921569, 'recall': 0.6037735849056604, 'f1': 0.6153846153846154, 'number': 53}, 'eval_LEGISLATION/ACT': {'precision': 0.8222222222222222, 'recall': 0.9673202614379085, 'f1': 0.8888888888888888, 'number': 153}, 'eval_LOCATION': {'precision': 0.7871485943775101, 'recall': 0.9560975609756097, 'f1': 0.8634361233480177, 'number': 205}, 'eval_MISCELLANEOUS': {'precision': 0.7052238805970149, 'recall': 0.7132075471698113, 'f1': 0.7091932457786115, 'number': 265}, 'eval_PERSON': {'precision': 0.8095238095238095, 'recall': 0.7870370370370371, 'f1': 0.7981220657276996, 'number': 108}, 'eval_overall_precision': 0.8117131910235359, 'eval_o

20

In [43]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu":"0"})

In [44]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [45]:
results = {}
trainer_all = {}
s = splits[3]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(ener7_ds, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_len


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([14]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([14, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2347/2347 [00:00<00:00, 8298.63 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_10204\17258847.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Business,Government,Legislation/act,Location,Miscellaneous,Person,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.071500,0.029262,"{'precision': 0.8650602409638555, 'recall': 0.9088607594936708, 'f1': 0.8864197530864197, 'number': 395}","{'precision': 0.8793103448275862, 'recall': 0.8360655737704918, 'f1': 0.8571428571428572, 'number': 61}","{'precision': 0.8666666666666667, 'recall': 0.8478260869565217, 'f1': 0.8571428571428571, 'number': 138}","{'precision': 0.8761061946902655, 'recall': 0.9428571428571428, 'f1': 0.9082568807339451, 'number': 105}","{'precision': 0.7172774869109948, 'recall': 0.845679012345679, 'f1': 0.7762039660056658, 'number': 162}","{'precision': 0.9565217391304348, 'recall': 0.9166666666666666, 'f1': 0.9361702127659574, 'number': 72}",0.845056,0.888532,0.866249,0.992853,0.992853,0.546473,0.992754
2,0.018700,0.019522,"{'precision': 0.9183673469387755, 'recall': 0.9113924050632911, 'f1': 0.914866581956798, 'number': 395}","{'precision': 0.8484848484848485, 'recall': 0.9180327868852459, 'f1': 0.8818897637795275, 'number': 61}","{'precision': 0.9424460431654677, 'recall': 0.9492753623188406, 'f1': 0.9458483754512634, 'number': 138}","{'precision': 0.8761061946902655, 'recall': 0.9428571428571428, 'f1': 0.9082568807339451, 'number': 105}","{'precision': 0.8070175438596491, 'recall': 0.8518518518518519, 'f1': 0.8288288288288288, 'number': 162}","{'precision': 0.9848484848484849, 'recall': 0.9027777777777778, 'f1': 0.9420289855072465, 'number': 72}",0.896515,0.909968,0.903191,0.994949,0.994949,0.553617,0.994837
3,0.010700,0.020734,"{'precision': 0.9228855721393034, 'recall': 0.9392405063291139, 'f1': 0.9309912170639898, 'number': 395}","{'precision': 0.8823529411764706, 'recall': 0.9836065573770492, 'f1': 0.9302325581395349, 'number': 61}","{'precision': 0.9148936170212766, 'recall': 0.9347826086956522, 'f1': 0.924731182795699, 'number': 138}","{'precision': 0.9017857142857143, 'recall': 0.9619047619047619, 'f1': 0.9308755760368663, 'number': 105}","{'precision': 0.8209876543209876, 'recall': 0.8209876543209876, 'f1': 0.8209876543209876, 'number': 162}","{'precision': 0.9577464788732394, 'recall': 0.9444444444444444, 'f1': 0.951048951048951, 'number': 72}",0.901674,0.923901,0.912652,0.995202,0.995202,0.698210,0.995129
4,0.006200,0.019709,"{'precision': 0.9419191919191919, 'recall': 0.9443037974683545, 'f1': 0.9431099873577751, 'number': 395}","{'precision': 0.8939393939393939, 'recall': 0.9672131147540983, 'f1': 0.9291338582677166, 'number': 61}","{'precision': 0.9629629629629629, 'recall': 0.9420289855072463, 'f1': 0.9523809523809523, 'number': 138}","{'precision': 0.9357798165137615, 'recall': 0.9714285714285714, 'f1': 0.9532710280373832, 'number': 105}","{'precision': 0.7944444444444444, 'recall': 0.8827160493827161, 'f1': 0.8362573099415205, 'number': 162}","{'precision': 0.9436619718309859, 'recall': 0.9305555555555556, 'f1': 0.9370629370629372, 'number': 72}",0.913271,0.936763,0.924868,0.995808,0.995808,0.702056,0.995737
5,0.003800,0.022645,"{'precision': 0.9393939393939394, 'recall': 0.9417721518987342, 'f1': 0.9405815423514539, 'number': 395}","{'precision': 0.8939393939393939, 'recall': 0.9672131147540983, 'f1': 0.9291338582677166, 'number': 61}","{'precision': 0.9485294117647058, 'recall': 0.9347826086956522, 'f1': 0.9416058394160585, 'number': 138}","{'precision': 0.9357798165137615, 'recall': 0.9714285714285714, 'f1': 0.9532710280373832, 'number': 105}","{'precision': 0.834319526627219, 'recall': 0.8703703703703703, 'f1': 0.851963746223565, 'number': 162}","{'precision': 0.9436619718309859, 'recall': 0.9305555555555556, 'f1': 0.9370629370629372, 'number': 72}",0.918691,0.932476,0.925532,0.995732,0.995732,0.701481,0.995645


F1 Macro: 0.7209232782530307
F1 Micro: 0.9953287908419387
F1 Weighted: 0.995285337316602
{'eval_loss': 0.021206628531217575, 'eval_BUSINESS': {'precision': 0.9172056921086675, 'recall': 0.9453333333333334, 'f1': 0.9310571240971767, 'number': 750}, 'eval_COURT': {'precision': 0.6, 'recall': 1.0, 'f1': 0.7499999999999999, 'number': 3}, 'eval_GOVERNMENT': {'precision': 0.8220338983050848, 'recall': 0.9065420560747663, 'f1': 0.8622222222222222, 'number': 107}, 'eval_LEGISLATION/ACT': {'precision': 0.9314516129032258, 'recall': 0.9203187250996016, 'f1': 0.9258517034068137, 'number': 251}, 'eval_LOCATION': {'precision': 0.91, 'recall': 0.9381443298969072, 'f1': 0.9238578680203046, 'number': 194}, 'eval_MISCELLANEOUS': {'precision': 0.8223684210526315, 'recall': 0.8223684210526315, 'f1': 0.8223684210526315, 'number': 304}, 'eval_PERSON': {'precision': 0.9888888888888889, 'recall': 0.956989247311828, 'f1': 0.9726775956284154, 'number': 93}, 'eval_overall_precision': 0.8981588032220944, 'eval_o

20

In [46]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu":"0"})

In [47]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [48]:
results = {}
trainer_all = {}
s = splits[4]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(ener7_ds, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_rare


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([14]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([14, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2347/2347 [00:00<00:00, 7138.91 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_10204\17258847.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Business,Court,Government,Legislation/act,Location,Miscellaneous,Person,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.061700,0.032271,"{'precision': 0.7407407407407407, 'recall': 0.9287925696594427, 'f1': 0.8241758241758242, 'number': 323}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2}","{'precision': 0.8148148148148148, 'recall': 0.6666666666666666, 'f1': 0.7333333333333333, 'number': 66}","{'precision': 0.9191919191919192, 'recall': 0.875, 'f1': 0.896551724137931, 'number': 104}","{'precision': 0.8878504672897196, 'recall': 0.9405940594059405, 'f1': 0.9134615384615384, 'number': 101}","{'precision': 0.8504672897196262, 'recall': 0.5909090909090909, 'f1': 0.6973180076628352, 'number': 154}","{'precision': 0.9302325581395349, 'recall': 0.9302325581395349, 'f1': 0.9302325581395349, 'number': 43}",0.811506,0.836066,0.823602,0.992335,0.992335,0.662982,0.992010
2,0.013600,0.023714,"{'precision': 0.8615819209039548, 'recall': 0.9442724458204335, 'f1': 0.9010339734121123, 'number': 323}","{'precision': 0.5, 'recall': 1.0, 'f1': 0.6666666666666666, 'number': 2}","{'precision': 0.8493150684931506, 'recall': 0.9393939393939394, 'f1': 0.8920863309352518, 'number': 66}","{'precision': 0.9215686274509803, 'recall': 0.9038461538461539, 'f1': 0.9126213592233009, 'number': 104}","{'precision': 0.9489795918367347, 'recall': 0.9207920792079208, 'f1': 0.934673366834171, 'number': 101}","{'precision': 0.855072463768116, 'recall': 0.7662337662337663, 'f1': 0.8082191780821918, 'number': 154}","{'precision': 0.9302325581395349, 'recall': 0.9302325581395349, 'f1': 0.9302325581395349, 'number': 43}",0.879310,0.900378,0.889720,0.994543,0.994543,0.675500,0.994368
3,0.008800,0.021959,"{'precision': 0.8936170212765957, 'recall': 0.9102167182662538, 'f1': 0.9018404907975459, 'number': 323}","{'precision': 0.5, 'recall': 0.5, 'f1': 0.5, 'number': 2}","{'precision': 0.9076923076923077, 'recall': 0.8939393939393939, 'f1': 0.900763358778626, 'number': 66}","{'precision': 0.9393939393939394, 'recall': 0.8942307692307693, 'f1': 0.9162561576354681, 'number': 104}","{'precision': 0.9320388349514563, 'recall': 0.9504950495049505, 'f1': 0.9411764705882353, 'number': 101}","{'precision': 0.8012048192771084, 'recall': 0.8636363636363636, 'f1': 0.83125, 'number': 154}","{'precision': 0.9302325581395349, 'recall': 0.9302325581395349, 'f1': 0.9302325581395349, 'number': 43}",0.887237,0.902900,0.895000,0.994594,0.994594,0.683421,0.994464
4,0.005300,0.022294,"{'precision': 0.8895522388059701, 'recall': 0.9226006191950464, 'f1': 0.9057750759878419, 'number': 323}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2}","{'precision': 0.9264705882352942, 'recall': 0.9545454545454546, 'f1': 0.9402985074626866, 'number': 66}","{'precision': 0.9595959595959596, 'recall': 0.9134615384615384, 'f1': 0.9359605911330049, 'number': 104}","{'precision': 0.9405940594059405, 'recall': 0.9405940594059405, 'f1': 0.9405940594059405, 'number': 101}","{'precision': 0.868421052631579, 'recall': 0.8571428571428571, 'f1': 0.8627450980392157, 'number': 154}","{'precision': 0.9302325581395349, 'recall': 0.9302325581395349, 'f1': 0.9302325581395349, 'number': 43}",0.906250,0.914250,0.910232,0.994975,0.994975,0.812422,0.994922
5,0.003400,0.021536,"{'precision': 0.9066265060240963, 'recall': 0.9318885448916409, 'f1': 0.9190839694656489, 'number': 323}","{'precision': 0.6666666666666666, 'recall': 1.0, 'f1': 0.8, 'number': 2}","{'precision': 0.9104477611940298, 'recall': 0.9242424242424242, 'f1': 0.9172932330827067, 'number': 66}","{'precision': 0.9504950495049505, 'recall': 0.9230769230769231, 'f1': 0.9365853658536586, 'number': 104}","{'precision': 0.9411764705882353, 'recall': 0.9504950495049505, 'f1': 0.9458128078817734, 'number': 101}","{'precision': 0.869281045751634, 'recall': 0.8636363636363636, 'f1': 0.8664495114006514, 'number': 154}","{'precision': 0.9302325581395349, 'recall': 0.93023255

F1 Macro: 0.5142577194317072
F1 Micro: 0.9868813867636831
F1 Weighted: 0.9864644856189444
{'eval_loss': 0.06744734942913055, 'eval_BUSINESS': {'precision': 0.8041666666666667, 'recall': 0.8717253839205059, 'f1': 0.8365843086259211, 'number': 1107}, 'eval_COURT': {'precision': 0.8888888888888888, 'recall': 0.8888888888888888, 'f1': 0.8888888888888888, 'number': 9}, 'eval_GOVERNMENT': {'precision': 0.8796992481203008, 'recall': 0.8297872340425532, 'f1': 0.854014598540146, 'number': 141}, 'eval_LEGISLATION/ACT': {'precision': 0.8609865470852018, 'recall': 0.8687782805429864, 'f1': 0.8648648648648648, 'number': 221}, 'eval_LOCATION': {'precision': 0.8363095238095238, 'recall': 0.8438438438438438, 'f1': 0.8400597907324365, 'number': 333}, 'eval_MISCELLANEOUS': {'precision': 0.6994106090373281, 'recall': 0.7021696252465484, 'f1': 0.7007874015748032, 'number': 507}, 'eval_PERSON': {'precision': 0.9246231155778895, 'recall': 0.92, 'f1': 0.9223057644110277, 'number': 200}, 'eval_overall_precisi

20

In [49]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu":"0"})

In [50]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [51]:
results = {}
trainer_all = {}
s = splits[5]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(ener7_ds, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: advs
Selecionando 2347 sentenças para teste…
  0 selecionadas…
  24 selecionadas…
  48 selecionadas…
  73 selecionadas…
  96 selecionadas…
  119 selecionadas…
  138 selecionadas…
  158 selecionadas…
  177 selecionadas…
  197 selecionadas…
  215 selecionadas…
  232 selecionadas…
  247 selecionadas…
  260 selecionadas…
  285 selecionadas…
  309 selecionadas…
  331 selecionadas…
  354 selecionadas…
  374 selecionadas…
  393 selecionadas…
  413 selecionadas…
  436 selecionadas…
  453 selecionadas…
  469 selecionadas…
  486 selecionadas…
  506 selecionadas…
  524 selecionadas…
  546 selecionadas…
  561 selecionadas…
  571 selecionadas…
  586 selecionadas…
  603 selecionadas…
  613 selecionadas…
  625 selecionadas…
  643 selecionadas…
  666 selecionadas…
  690 selecionadas…
  710 selecionadas…
  733 selecionadas…
  757 selecionadas…
  781 selecionadas…
  799 selecionadas…
  818 selecionadas…
  839 selecionadas…
  856 selecionadas…
  876 selecionadas…
  898 selecionadas…


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([14]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([14, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2348/2348 [00:00<00:00, 6684.10 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_10204\17258847.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Business,Court,Government,Legislation/act,Location,Miscellaneous,Person,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.075900,0.031389,"{'precision': 0.8933649289099526, 'recall': 0.854875283446712, 'f1': 0.87369640787949, 'number': 441}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2}","{'precision': 0.8, 'recall': 0.9090909090909091, 'f1': 0.8510638297872342, 'number': 66}","{'precision': 0.8245614035087719, 'recall': 0.8392857142857143, 'f1': 0.8318584070796461, 'number': 112}","{'precision': 0.911504424778761, 'recall': 0.8956521739130435, 'f1': 0.9035087719298245, 'number': 115}","{'precision': 0.6096491228070176, 'recall': 0.776536312849162, 'f1': 0.683046683046683, 'number': 179}","{'precision': 0.9382716049382716, 'recall': 0.9382716049382716, 'f1': 0.9382716049382716, 'number': 81}",0.822222,0.854418,0.838011,0.990695,0.990695,0.743678,0.990798
2,0.019600,0.020925,"{'precision': 0.8950749464668094, 'recall': 0.9478458049886621, 'f1': 0.9207048458149779, 'number': 441}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2}","{'precision': 0.8676470588235294, 'recall': 0.8939393939393939, 'f1': 0.8805970149253731, 'number': 66}","{'precision': 0.9130434782608695, 'recall': 0.9375, 'f1': 0.9251101321585903, 'number': 112}","{'precision': 0.8728813559322034, 'recall': 0.8956521739130435, 'f1': 0.8841201716738197, 'number': 115}","{'precision': 0.7696078431372549, 'recall': 0.8770949720670391, 'f1': 0.8198433420365535, 'number': 179}","{'precision': 0.925, 'recall': 0.9135802469135802, 'f1': 0.9192546583850932, 'number': 81}",0.870968,0.921687,0.895610,0.994490,0.994490,0.760495,0.994470
3,0.010700,0.017098,"{'precision': 0.9120171673819742, 'recall': 0.963718820861678, 'f1': 0.9371554575523704, 'number': 441}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2}","{'precision': 0.9264705882352942, 'recall': 0.9545454545454546, 'f1': 0.9402985074626866, 'number': 66}","{'precision': 0.8983050847457628, 'recall': 0.9464285714285714, 'f1': 0.9217391304347826, 'number': 112}","{'precision': 0.9174311926605505, 'recall': 0.8695652173913043, 'f1': 0.8928571428571428, 'number': 115}","{'precision': 0.9064327485380117, 'recall': 0.8659217877094972, 'f1': 0.8857142857142858, 'number': 179}","{'precision': 0.9390243902439024, 'recall': 0.9506172839506173, 'f1': 0.9447852760736196, 'number': 81}",0.913386,0.931727,0.922465,0.995971,0.995971,0.935944,0.995958
4,0.006300,0.018064,"{'precision': 0.9276315789473685, 'recall': 0.9591836734693877, 'f1': 0.94314381270903, 'number': 441}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2}","{'precision': 0.8857142857142857, 'recall': 0.9393939393939394, 'f1': 0.9117647058823529, 'number': 66}","{'precision': 0.9469026548672567, 'recall': 0.9553571428571429, 'f1': 0.9511111111111111, 'number': 112}","{'precision': 0.9122807017543859, 'recall': 0.9043478260869565, 'f1': 0.9082969432314411, 'number': 115}","{'precision': 0.9390243902439024, 'recall': 0.8603351955307262, 'f1': 0.8979591836734694, 'number': 179}","{'precision': 0.9382716049382716, 'recall': 0.9382716049382716, 'f1': 0.9382716049382716, 'number': 81}",0.928000,0.931727,0.929860,0.996075,0.996075,0.936136,0.996060
5,0.004800,0.018492,"{'precision': 0.9259259259259259, 'recall': 0.963718820861678, 'f1': 0.9444444444444445, 'number': 441}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2}","{'precision': 0.9117647058823529, 'recall': 0.9393939393939394, 'f1': 0.9253731343283583, 'number': 66}","{'precision': 0.9473684210526315, 'recall': 0.9642857142857143, 'f1': 0.9557522123893805, 'number': 112}","{'precision': 0.9130434782608695, 'recall': 0.9130434782608695, 'f1': 0.9130434782608695, 'number': 115}","{'precision': 0.9281437125748503, 'recall': 0.8659217877094972, 'f1': 0.8959537572254336, 'number': 179}","{'precision': 0.9382716049382716, 'recall': 0.9382716049382716, 'f1': 0.9382716049382716, 'number': 81}"

c:\Users\user\miniconda3\envs\lora-gpu\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


F1 Macro: 0.709947166077603
F1 Micro: 0.995116487014965
F1 Weighted: 0.9948871235907478
{'eval_loss': 0.03629439324140549, 'eval_BUSINESS': {'precision': 0.8502304147465438, 'recall': 0.8482758620689655, 'f1': 0.849252013808976, 'number': 435}, 'eval_COURT': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}, 'eval_GOVERNMENT': {'precision': 0.8932038834951457, 'recall': 0.9292929292929293, 'f1': 0.9108910891089108, 'number': 99}, 'eval_LEGISLATION/ACT': {'precision': 0.9551282051282052, 'recall': 0.952076677316294, 'f1': 0.9536, 'number': 313}, 'eval_LOCATION': {'precision': 0.9020979020979021, 'recall': 0.9148936170212766, 'f1': 0.9084507042253521, 'number': 141}, 'eval_MISCELLANEOUS': {'precision': 0.7871287128712872, 'recall': 0.7429906542056075, 'f1': 0.764423076923077, 'number': 214}, 'eval_PERSON': {'precision': 0.9387755102040817, 'recall': 0.9928057553956835, 'f1': 0.965034965034965, 'number': 139}, 'eval_overall_precision': 0.883668903803132, 'eval_overall_recall': 0.8

20

In [52]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu":"0"})

In [53]:
df = ner_collector.to_dataframes()

In [54]:
df

(       split  eval_loss  overall_precision  overall_recall  overall_f1  \
 0        loc   0.075636           0.797276        0.810921    0.804040   
 1    reverse   0.498730           0.578115        0.503396    0.538175   
 2   semantic   0.080664           0.811713        0.861708    0.835964   
 3   heur_len   0.021207           0.898159        0.917156    0.907558   
 4  heur_rare   0.067447           0.806056        0.835187    0.820363   
 5       advs   0.036294           0.883669        0.882353    0.883010   
 
    overall_accuracy  f1_micro  f1_macro  f1_weighted  runtime_s  \
 0          0.984994  0.984994  0.551618     0.984306     7.2170   
 1          0.937248  0.937248  0.329959     0.925181    30.0828   
 2          0.986599  0.986599  0.733467     0.986283     9.0583   
 3          0.995329  0.995329  0.720923     0.995285    17.3514   
 4          0.986881  0.986881  0.514258     0.986464    21.4706   
 5          0.995116  0.995116  0.709947     0.994887    18.3491 

In [55]:
nome_modelo = MODEL_NAME.split("/")[1]

In [56]:
ner_collector.to_csv(base_name=nome_modelo)

('albertina-base_overall_metrics.csv', 'albertina-base_label_metrics.csv')